In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Rescaling
from tensorflow.keras.models import Model
import zipfile

print("1. Unzipping dataset...")
with zipfile.ZipFile('/content/Dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

BATCH_SIZE = 32
IMG_SIZE = (224, 224)
train_dir = '/content/dataset/train'
test_dir = '/content/dataset/test'

print("2. Loading images into TensorFlow...")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE
)

print("3. Building MobileNetV2 brain with internal Rescaling...")
# Load pre-trained base
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False 

# THIS IS THE FIX: Build a pipeline that scales 0-255 down to -1 to 1 automatically
inputs = tf.keras.Input(shape=(224, 224, 3))
x = Rescaling(scale=1./127.5, offset=-1)(inputs) # The magic math layer
x = base_model(x)
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(4, activation='softmax')(x) 

model = Model(inputs=inputs, outputs=predictions)

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("4. Training the model...")
history = model.fit(
    train_dataset,
    epochs=10,
    validation_data=validation_dataset
)

print("5. Converting to TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('grid_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("✅ DONE! You can now download the FIXED 'grid_model.tflite'.")